# Dose-level Loewe additivity — the additivity *reference* as a model-selection axis

v0.23 combined two **effect magnitudes** under effect-level nulls (HSA, Bliss-additive, Greco).
Loewe additivity instead combines two **doses** via the dose-response (exposure-response) curves,
solving the isobole $d_A/D_A(E) + d_B/D_B(E) = 1$, where $D_x(E)$ is the dose of drug $x$ alone
producing effect $E$.

Loewe is the only 'no-interaction' reference that satisfies the **sham-combination identity** — a
drug combined with itself is *exactly* additive — which Bliss/effect-additivity fails for any
saturating curve. So the additivity reference is a model-selection axis, and it propagates to OS.
*Population/regimen level only; the reference is a declared modeling choice, never fitted.*

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.interaction import er_curve, loewe_effect, combine_doses, compare_additivity_references

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
ER_A = 'exposure_response.emax_generic'          # E = 1.4*C/(150+C)
ER_B = 'exposure_response.dacomitinib_egfr.emax' # E = 1.8*C/(90+C)

## The sham-combination identity (what makes Loewe principled)

Combine drug A with *itself*, splitting a dose. Loewe must return the single-agent effect at the
summed dose, $f(d_A+d_B)$ — and it does, exactly. Bliss (effect-additive) does not: $f(d_A)+f(d_B)$
overshoots, because you are climbing the *same* saturating curve twice.

In [ ]:
ca = er_curve(ds, ER_A)
da, db = 100.0, 200.0
loewe_self = loewe_effect(da, db, curve_a=ca, curve_b=ca)
bliss_self = ca.forward(da) + ca.forward(db)
print(f'Loewe(A,A) = {loewe_self:.5f}   f(d_A+d_B) = {ca.forward(da+db):.5f}   (match: {np.isclose(loewe_self, ca.forward(da+db))})')
print(f'Bliss(A,A) = {bliss_self:.5f}   != f(d_A+d_B)  ->  effect-additivity fails the sham test')

## Three references, three combined effects, three survival curves

For the same dose pair, HSA (conservative), Loewe (self-consistent), and Bliss (effect-additive,
overstates) give different combined effects — and feeding each through the existing TGI→survival
chain gives different OS.

In [ ]:
for ref in ('hsa', 'loewe', 'bliss'):
    e = combine_doses(ds, 150.0, 90.0, er_a=ER_A, er_b=ER_B, reference=ref)
    print(f'{ref:6s} combined effect = {e:.3f}')
cmp = compare_additivity_references(ds, 'resistance.claret_2009.tgi', context=ctx,
                                    dose_a=150.0, dose_b=90.0, er_a=ER_A, er_b=ER_B)
print()
for ref, mos in cmp.median_os.items():
    print(f'{ref:6s} median OS = {mos:.0f} wk')
print(f'\nOS divergence across references = {cmp.os_divergence:.3f}')

## The takeaway

Combination oncology has no single 'no-interaction' reference — HSA, Bliss, and Loewe are three
defensible nulls that disagree, and the disagreement *grows with dose* as the saturating curves are
climbed (Bliss can even exceed either drug's maximum effect). Loewe is the only one that is
self-consistent under sham combination. Onkos makes the reference a declared, swappable choice and
quantifies its OS consequence; it crowns no winner and estimates no synergy. **No therapy ranking,
no individual prediction; the underlying model's tier governs and cannot be raised.**